# redispatch-break: forecasting Spain's Phase-I upward redispatch energy across the 2025 blackout
**Branch:** `feature/redispatch-break` | **Gate:** pre-registered in README.md (frozen 2026-11-13)
**One-Click Execution:** Runtime -> Run all (CPU runtime, no GPU needed). Mounts Drive, clones, installs, runs pytest,
downloads the public REE files (about 200 MB, kept on Drive), runs `data`, `dev`, `test`, prints RESULTS.md.
Re-running reuses the downloaded files. If REE refuses downloads from Colab, copy the Mac's `data/raw` folder to
`MyDrive/moonshot_results/redispatch-break/data/raw` and run again.
If any step fails, or a stage runs longer than 4 hours, Drive is flushed and the runtime is released.

In [ ]:
# Step 1: Mount Drive; downloads and results live there
import os  # first cell: the Drive prompt appears right away
from google.colab import drive
drive.mount("/content/drive")
RES = "/content/drive/MyDrive/moonshot_results/redispatch-break"
os.environ["RB_DATA"] = RES + "/data"
os.environ["OMP_NUM_THREADS"] = "1"  # Colab CPU has 2 cores: one thread per worker, or the run crawls
os.makedirs(os.environ["RB_DATA"], exist_ok=True)
print("Results dir:", RES, "| existing:", sorted(os.listdir(RES)))

In [ ]:
# Step 2: Credit guard: on any error, flush Drive and release the runtime
from IPython import get_ipython

def auto_teardown():
    print("\n[Credit Guard] Flushing Drive and releasing the runtime...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def _on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()

get_ipython().set_custom_exc((Exception,), _on_error)
print("[Ready] The runtime is released on any error, on a stage time-out, and at the end.")

In [ ]:
# Step 2b: Clone the repository and install
import sys, subprocess
repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/redispatch-break"
target_dir = "/content/redispatch-break"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
os.chdir(target_dir)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(subprocess.run([sys.executable, "-c", "import os, numpy, pandas, sklearn; print('cpus', os.cpu_count(), '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)"], capture_output=True, text=True).stdout)

In [ ]:
# Step 3: Pre-flight (no network): parsers, no-look-ahead checks, metrics, gate rules. Aborts if anything fails.
_t = subprocess.run([sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider", "test_rb.py"], capture_output=True, text=True)  # Colab hides child output
print(_t.stdout[-20000:]); print(_t.stderr[-5000:])
open(RES + "/pytest_log.txt", "w").write(_t.stdout + _t.stderr)
if _t.returncode:
    raise RuntimeError("pytest failed -> run aborted")
print("[Pre-flight] all checks passed.")

In [ ]:
# Step 4: data (download + daily table + integrity check), dev (pre-blackout only), test (frozen gate)
# Each stage is killed after RB_MAX_HOURS (default 4); the error then releases the runtime.
import signal, threading, time
MAX_HOURS = float(os.environ.get("RB_MAX_HOURS", "4"))
for stage in ["data", "dev", "test"]:
    print(f"\n===== {stage} =====", flush=True)
    t0 = time.time()
    p = subprocess.Popen([sys.executable, "-u", "-W", "ignore", "run_all.py", stage], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, start_new_session=True)
    killer = threading.Timer(MAX_HOURS * 3600, lambda: os.killpg(p.pid, signal.SIGKILL))  # the stage and its workers
    killer.start()
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    killer.cancel()
    print(f"[{stage}] {(time.time() - t0) / 60:.1f} min", flush=True)
    if rc:
        raise RuntimeError(f"run_all.py {stage} failed or timed out (exit {rc})")

In [ ]:
# Step 5: Results (also copied to Drive), then release the runtime
import shutil
shutil.copy("RESULTS.md", RES + "/RESULTS.md")
print(open("RESULTS.md").read())
auto_teardown()